# Gemini & Gemma Bike Helmet Detector
This notebook detects whether bike riders are wearing helmets in images of bikes located in the `Vehicle Snapshots` structured directories. It utilizes the modern `google-genai` SDK and targets Gemma 4 (31B), maximizing both detection accuracy and processing speed based on the 15 RPM limit.

In [1]:
# Install the new google-genai package and ipywidgets (to fix the IProgress/tqdm warning)
!pip install -q google-genai ipywidgets

In [ ]:
import os
import glob
import csv
from time import sleep

try:
    from google import genai
    from google.genai import errors
except ImportError:
    print("Please run the cell above to install google-genai")
    raise

# Initialize the client with your provided API key
client = genai.Client(api_key='API_KEY')

# Targeting Gemma 4 31B specifically for its potentially superior detection capabilities.
model_id = 'gemma-4-31b-it'

In [3]:
# Updated directory to reflect the new structure
directories = [
    os.path.join('Vehicle Snapshots', 'Bikes (Number-Plates On The Rear Mud-Guard)')
]
csv_filename = "bike_helmets.csv"
results = []

# Prompt instruction for the Gemma Model
prompt = "Analyze this image of a bike rider. Does the rider appear to be wearing a helmet? Provide only a concise 'YES' or 'NO' answer. If unsure or if the rider is not visible, return 'UNKNOWN'."

for directory in directories:
    if not os.path.isdir(directory):
        print(f"Directory {directory} not found, skipping.")
        continue
        
    print(f"Scanning Directory: {directory}...")
    for root, dirs, files in os.walk(directory):
        for file in files:
            # Filter condition: Image contains 'bike' in its name, case-insensitive
            if 'bike' in file.lower() and file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                file_path = os.path.join(root, file)
                print(f"Loading {file_path}...")
                
                try:
                    # Upload image to AI platform using the new SDK
                    uploaded_img = client.files.upload(file=file_path)
                    
                    # Logic with Exponential Backoff specifically for 503 Server Overloads or rate limits
                    max_retries = 3
                    response = None
                    for attempt in range(max_retries):
                        try:
                            response = client.models.generate_content(
                                model=model_id,
                                contents=[uploaded_img, prompt]
                            )
                            break  # Success! Break out of the retry loop
                        except Exception as e:
                            if ("503" in str(e) or "429" in str(e)) and attempt < max_retries - 1:
                                # Model is experiencing high demand or unexpected rate limits.
                                print(f"  -> Limit hit ({e}). Retrying attempt {attempt + 1} after 15 seconds...")
                                sleep(15)
                            else:
                                raise e # Error wasn't a catchable limit or we ran out of retries
                        
                    helmet_text = response.text.strip()
                    if "yes" in helmet_text.lower():
                        helmet_status = "YES"
                    elif "no" in helmet_text.lower():
                        helmet_status = "NO"
                    else:
                        helmet_status = helmet_text.upper()
                        
                    print(f"Helmet Status: {helmet_status}")
                    
                    results.append({'Image Name': file, 'Directory': directory, 'Helmet': helmet_status})
                    
                    # Delete file context to manage user cloud storage quotas
                    client.files.delete(name=uploaded_img.name)
                    
                    # Reduced Rate Limit Spacing for 15 RPM Capability:
                    # 60 seconds / 15 allowed requests = 4 seconds required between processing calls.
                    # We use 4.2 seconds to guarantee we stay comfortably under the 15 RPM strict boundary.
                    sleep(4.2)
                    
                except Exception as e:
                    print(f"Failed to process {file_path}: {e}")
                    results.append({'Image Name': file, 'Directory': directory, 'Helmet': 'ERROR'})

print("Finished scanning files.")

Scanning Directory: Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)...
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00217_bike.jpg...
Helmet Status: YES
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00272_bike.jpg...
Helmet Status: YES
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00304_bike.jpg...
Helmet Status: NO
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00336_bike.jpg...
Helmet Status: YES
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00412_bike.jpg...
Helmet Status: NO
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00419_bike.jpg...
Helmet Status: NO
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00509_bike.jpg...
Helmet Status: YES
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00607_bike.jpg...
Helmet Status: NO
Loading Vehicle Snapshots\Bikes (Number-Plates On The R

In [4]:
# Save extracted info in CSV format
with open(csv_filename, mode='w', newline='', encoding='utf-8') as f:
    # Standardized headers
    writer = csv.DictWriter(f, fieldnames=["Image Name", "Directory", "Helmet"])
    writer.writeheader()
    writer.writerows(results)

print(f"CSV data accurately written and saved to {csv_filename}!")

CSV data accurately written and saved to bike_helmets.csv!
